# 🤗 GDG AI Track Day 1 — Hugging Face Hands-on

> **GDG on Campus SKKU 2026 — AI Track Week 1**
> 텍스트 임베딩·분류 & 텍스트 생성 (90분)
> 발표자: 안호성 ([@BetaTester772](https://github.com/BetaTester772))

오늘 우리가 직접 돌려볼 5개의 실습 + 보너스 1개:

| # | 실습 | 모델 | task |
|---|---|---|---|
| ① | 문장 유사도 | `all-MiniLM-L6-v2` | embedding |
| ② | 감정 분석 | `distilbert-sst2` (자동 선택) | sentiment |
| ③ | Zero-shot 분류 | `bart-large-mnli` | zero-shot |
| ④ | 텍스트 생성 | `SmolLM2-360M-Instruct` | text-gen |
| ⑤ | 요약 | `distilbart-cnn` (자동 선택) | summarization |
| 🎁 | 보너스: 한국어 감정 분석 | `multilingual-sentiment` | sentiment (KR) |

> 💡 **시작하기 전에**: 메뉴에서 `Runtime → Change runtime type → T4 GPU` 를 선택해주세요.
> GPU 없이도 돌아가지만 일부 모델은 매우 느려집니다.

## 🔧 Section 0. Setup

> 📊 발표 슬라이드 1–3 참조

먼저 GPU가 잘 잡혔는지 확인하고, 필요한 라이브러리를 설치할게요.

In [ ]:
# GPU 정보 출력 (GPU 런타임이 아니면 에러 메시지가 나오지만 정상)
!nvidia-smi || true

In [ ]:
# 핵심 라이브러리 설치
# transformers>=4.45 — SmolLM2 chat template 안정성 위해
!pip install -q "transformers>=4.45" datasets sentence-transformers

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import gc
import torch
import pandas as pd
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from datasets import load_dataset

# 🎯 GPU가 잡혔으면 device=0, 아니면 device=-1 (CPU) — 모든 pipeline에서 재사용
device = 0 if torch.cuda.is_available() else -1
print(f"device = {device}  ({'GPU' if device == 0 else 'CPU'})")
if device == -1:
    print("💡 GPU 안 켜진 학생: Runtime → Change runtime type → T4 GPU 권장")

> ⏱️ **참고**: 각 모델은 첫 실행 시 다운로드에 1~2분 걸릴 수 있어요.
> 두 번째부터는 캐시 덕분에 빠르게 로드됩니다.

---

## 📍 실습 ① 문장 유사도 계산

> 📊 발표 슬라이드 4–11 참조

**이번에 할 일**: 문장들을 384차원 벡터로 바꾸고, **코사인 유사도**로 의미가 비슷한 문장끼리 짝지어보기.

**모델**: `sentence-transformers/all-MiniLM-L6-v2` (~80MB)
**예상 시간**: 약 10분

> 💡 슬라이드 10번의 "2D로 압축한 임베딩 공간" 그림을 코드로 직접 재현해봅니다.

In [ ]:
# 임베딩 모델 로드
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda" if device == 0 else "cpu")
print("✅ 모델 로드 완료. 출력 벡터 차원:", embedder.get_sentence_embedding_dimension())

In [ ]:
# 의미가 비슷한 3개 그룹으로 묶이는지 살펴봅시다
sentences = [
    "A dog is running in the park",       # animals
    "The puppy is chasing a ball",         # animals
    "I love eating fresh pizza",           # food
    "This pasta dish is delicious",        # food
    "The stock market crashed today",      # finance
    "Interest rates went up sharply",      # finance
]

# 6개 문장을 한 번에 인코딩 → shape: (6, 384)
embeddings = embedder.encode(sentences)
print("임베딩 행렬 shape:", embeddings.shape)

In [ ]:
# 모든 쌍의 코사인 유사도 행렬 계산
sim_matrix = cosine_similarity(embeddings)

# 보기 좋게 DataFrame으로
labels = [f"S{i+1}" for i in range(len(sentences))]
df = pd.DataFrame(sim_matrix, index=labels, columns=labels).round(3)
print("=== 코사인 유사도 행렬 ===\n")
print(df)
print("\n=== 문장 ===")
for i, s in enumerate(sentences):
    print(f"S{i+1}: {s}")

**🔍 결과 해석 포인트**
- S1↔S2 (강아지 둘), S3↔S4 (음식 둘), S5↔S6 (금융 둘) → 같은 그룹은 **0.5 이상**
- 다른 그룹끼리는 **0.2 이하**로 떨어지는 게 보일 거예요.

**🎯 본인 차례**: 위 `sentences` 리스트에 본인이 만든 문장 하나를 추가하고 어느 그룹과 가장 유사한지 확인해보세요.

---

## 📍 실습 ② 감정 분석 (Sentiment Analysis)

> 📊 발표 슬라이드 12–13 참조

**Pipeline API의 마법**: `pipeline("sentiment-analysis")` 한 줄로 모델 로드 + 추론까지 끝납니다.

> 💡 모델명을 안 주면 HF가 `distilbert-base-uncased-finetuned-sst-2-english`를 자동 선택합니다.
> (영어 영화 리뷰 데이터인 SST-2로 fine-tune된 distilbert.)

**예상 시간**: 약 10분

In [ ]:
# Auto-picks: distilbert-base-uncased-finetuned-sst-2-english
classifier = pipeline("sentiment-analysis", device=device)
print("✅ 자동 선택된 모델:", classifier.model.name_or_path)

In [ ]:
# 짧은 문장으로 먼저 확인
texts = [
    "I absolutely loved this movie, what a masterpiece!",
    "Worst film I've ever seen. Total waste of time.",
    "It was okay I guess, not great not terrible.",
    "The acting was brilliant and the soundtrack moved me to tears.",
    "I want my money back.",
]
for t in texts:
    r = classifier(t)[0]
    print(f"[{r['label']:8s}  score={r['score']:.3f}]  {t}")

In [ ]:
# 진짜 IMDb 데이터에서 10개 샘플 뽑아 분류
ds = load_dataset("stanfordnlp/imdb", split="test[:10]")
for i, row in enumerate(ds):
    review = row["text"][:200].replace("<br />", " ")  # 200자로 자르기
    truth = "POSITIVE" if row["label"] == 1 else "NEGATIVE"
    pred = classifier(review)[0]
    match = "✅" if pred["label"] == truth else "❌"
    print(f"{match} truth={truth:8s}  pred={pred['label']:8s}({pred['score']:.2f})")
    print(f"   {review[:120]}...\n")

**🔍 관찰 포인트**
- `score` (confidence) 는 0~1 사이. 1에 가까울수록 모델이 확신하는 결과예요.
- 한국어를 넣으면 어떻게 될까요? → **거의 작동하지 않습니다**. 영어 데이터로만 학습했거든요. 보너스 섹션에서 해결!

---

## 📍 실습 ③ Zero-shot 분류

> 📊 발표 슬라이드 14–17 참조

**Zero-shot이란?** 학습 데이터 없이, 분류하고 싶은 **라벨만 자연어로 정의**하면 바로 분류해주는 기법.

**원리** (슬라이드 16–17): NLI(Natural Language Inference) 모델을 재활용
1. 각 라벨을 "This text is about ___" 같은 가설 문장으로 변환
2. 원본 + 가설을 NLI 모델에 입력 → entailment 점수가 가장 높은 라벨 선택

**모델**: `facebook/bart-large-mnli` (~1.6GB, 다운로드 ~45초)
**데이터**: AG News 헤드라인
**예상 시간**: 약 5분

In [ ]:
# 메모리 정리 (Section 2의 distilbert는 작아서 굳이 필요 없지만 습관 들이기)
del classifier
gc.collect()
if device == 0:
    torch.cuda.empty_cache()

zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device)
print("✅ BART-MNLI 로드 완료")

In [ ]:
# AG News 테스트셋에서 헤드라인 5개
ds = load_dataset("fancyzhx/ag_news", split="test[:5]")
candidate_labels = ["technology", "sports", "business", "politics"]

for row in ds:
    text = row["text"]
    result = zero_shot(text, candidate_labels)
    top_label = result["labels"][0]
    top_score = result["scores"][0]
    print(f"[{top_label:12s} {top_score:.2f}]  {text[:80]}...")

**🎯 본인 차례**: `candidate_labels`를 자유롭게 바꿔보세요.
- 예: `["happy news", "sad news", "boring news"]`
- 예: `["about Apple", "about Google", "about Microsoft"]`

자연어로 라벨을 적기만 하면 **별도 학습 없이** 분류가 됩니다. 이게 Zero-shot의 힘이에요.

---

## ☕ 5분 휴식

여기까지 잘 따라오셨다면 5분 쉬어가세요. 다음 파트는 텍스트 **생성**입니다.

> 💡 휴식 동안 BART 모델을 메모리에서 내려서 다음 섹션 준비를 할게요.

---

# Part 2. Text Generation 🚀

## 📍 실습 ④ 텍스트 생성 (디코딩 전략 비교)

> 📊 발표 슬라이드 19–24 참조

**LM의 핵심 원리** (슬라이드 20): 직전 토큰까지를 보고 **다음 토큰의 확률 분포**를 계산.

**디코딩 파라미터** (슬라이드 24의 그림 그대로 재현):
- `temperature` — 낮을수록 결정적(반복적), 높을수록 창의적(영뚱)
- `top_p` — 누적 확률 p까지의 후보만 사용
- `max_new_tokens` — 생성 길이 제한

오늘은 **같은 프롬프트**에 `temperature`만 0.1 / 0.7 / 1.5 로 바꿔보며 출력을 비교합니다.

**모델**: `HuggingFaceTB/SmolLM2-360M-Instruct` (~720MB)

In [ ]:
# BART 정리 후 SmolLM2 로드 (T4 16GB 메모리 절약)
del zero_shot
gc.collect()
if device == 0:
    torch.cuda.empty_cache()

generator = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
    device=device,
    torch_dtype=torch.float16 if device == 0 else torch.float32,
)
print("✅ SmolLM2-360M-Instruct 로드 완료")

In [ ]:
# 슬라이드 24와 동일한 프롬프트 — chat template 형식
messages = [
    {"role": "user", "content": "Tell me a short story about a robot"},
]

# temperature 0.1 / 0.7 / 1.5 를 비교
for temp in [0.1, 0.7, 1.5]:
    print(f"\n{'='*60}\n🌡️  temperature = {temp}\n{'='*60}")
    out = generator(
        messages,
        max_new_tokens=120,
        do_sample=True,        # ⚠️ temperature는 do_sample=True 일 때만 작동
        temperature=temp,
        top_p=0.9,
        return_full_text=False,
    )
    print(out[0]["generated_text"].strip())

**🔍 관찰 포인트** (슬라이드 24의 A/B/C 시각화와 같은 패턴)
- **0.1** → 일관성 있지만 단조로움 (거의 greedy)
- **0.7** → 자연스러운 다양성, 가장 실용적
- **1.5** → 창의적이지만 가끔 영뚱하거나 문법이 깨짐

**🎯 본인 차례**: `messages` 의 content 를 본인이 좋아하는 프롬프트로 바꿔보세요.
- 예: `"Write a haiku about coding"`, `"Explain quantum computing to a 5 year old"`

---

## 📍 실습 ⑤ 텍스트 요약

> 📊 발표 슬라이드 25 참조

**Seq2Seq 모델** = 입력 텍스트를 다른 텍스트로 변환하는 구조 (encoder + decoder).
요약은 "긴 입력 → 짧은 출력" 매핑이 명확해서 Seq2Seq가 매우 강해요.

> 💡 모델명을 안 주면 HF가 `sshleifer/distilbart-cnn-12-6`를 자동 선택합니다.
> (CNN/DailyMail 뉴스 요약 데이터로 학습된 distilled BART.)

**예상 시간**: 약 15분

In [ ]:
# SmolLM2 정리 후 요약 모델 로드
del generator
gc.collect()
if device == 0:
    torch.cuda.empty_cache()

# Auto-picks: sshleifer/distilbart-cnn-12-6
summarizer = pipeline("summarization", device=device)
print("✅ 자동 선택된 모델:", summarizer.model.name_or_path)

In [ ]:
# Source: facebook/bart-large-cnn 모델 카드의 공식 example 텍스트
ARTICLE = """
The tower is 324 metres (1,063 ft) tall, about the same height as an 81-storey building,
and the tallest structure in Paris. Its base is square, measuring 125 metres (410 ft) on each side.
During its construction, the Eiffel Tower surpassed the Washington Monument to become the tallest
man-made structure in the world, a title it held for 41 years until the Chrysler Building in
New York City was finished in 1930. It was the first structure to reach a height of 300 metres.
Due to the addition of a broadcasting aerial at the top of the tower in 1957, it is now taller
than the Chrysler Building by 5.2 metres (17 ft). Excluding transmitters, the Eiffel Tower is
the second tallest free-standing structure in France after the Millau Viaduct.
"""

# 1) 표준 길이 요약
summary_long = summarizer(ARTICLE, max_length=130, min_length=30, do_sample=False)[0]["summary_text"]
print("=== max_length=130 ===")
print(summary_long)

# 2) 더 짧게
summary_short = summarizer(ARTICLE, max_length=50, min_length=15, do_sample=False)[0]["summary_text"]
print("\n=== max_length=50 ===")
print(summary_short)

**🎯 본인 차례**: 본인이 가져온 뉴스 기사 한 단락이나 위키피디아 문단을 `ARTICLE` 변수에 넣어보세요.
`max_length` 와 `min_length` 를 바꿔가며 요약 품질이 어떻게 달라지는지 관찰해보세요.

---

## 🎁 보너스: 한국어 감정 분석

> 📊 발표 슬라이드 26 참조

Section 2 의 영어 감정분석이 한국어에 안 됐던 거 기억하시죠?
**모델만 바꾸면 됩니다.** 같은 `pipeline("sentiment-analysis")` 코드, 다른 모델.

In [ ]:
# 요약 모델 정리 후 한국어 감정 분류기 로드
del summarizer
gc.collect()
if device == 0:
    torch.cuda.empty_cache()

classifier = pipeline(
    "sentiment-analysis",
    model="tabularisai/multilingual-sentiment-analysis",
    device=device,
)

korean_reviews = [
    "이 영화 진짜 재미있어요!",
    "별로였어요... 시간 낭비.",
    "평범했어요. 그냥 그래요.",
    "강력 추천! 인생 영화입니다.",
    "환불하고 싶네요.",
]
for r in korean_reviews:
    res = classifier(r)[0]
    print(f"[{res['label']:10s} {res['score']:.2f}]  {r}")

**💡 핵심 통찰**: 같은 코드, 다른 모델 = **HF 생태계의 힘**.
HF Hub에 가서 본인이 풀고 싶은 문제에 맞는 모델을 검색하기만 하면 됩니다.

---

## 🏁 마무리

> 📊 발표 슬라이드 27 참조

오늘 우리가 한 일:
1. 문장을 **384차원 벡터**로 바꾸고 코사인 유사도로 의미 비교
2. **Pipeline API** 한 줄로 감정 분류 (auto-pick의 마법)
3. **Zero-shot 분류** — 학습 없이 라벨만 정의해서 분류
4. **LLM 디코딩 파라미터** (temperature) 직접 체험
5. **Seq2Seq 요약** 으로 긴 글을 짧게
6. 🎁 모델만 바꿔서 **한국어**로 확장

### 여기서부터는 어떻게?

- 🔍 **HF Hub 탐색** — [huggingface.co](https://huggingface.co) 에서 task별로 모델 찾기
- 🛠️ **Fine-tuning** — 본인 데이터로 모델 미세조정 (Trainer API)
- 🌐 **Spaces** — Gradio로 5분만에 데모 웹앱 배포
- 📚 **추천 경로** — [HF Course (무료)](https://huggingface.co/learn)

**Happy Hugging 🤗**